In [2]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
LANG = "fr_FR"

BASE_DIR = r"D:\lol draft analyzer\part 2\data champions"
TGZ_DIR = os.path.join(BASE_DIR, "tgz")
JSON_DIR = os.path.join(BASE_DIR, "collect")
CSV_DIR = os.path.join(BASE_DIR, "csv")

os.makedirs(TGZ_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

# ================== UTILS ==================
def dump(value):
    """Serialize dict/list for CSV"""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

# ================== SPELL FLATTENER ==================
def flatten_spell(spell, prefix):
    row = {}
    for k, v in spell.items():
        row[f"{prefix}_{k}"] = dump(v)
    return row

# ================== CHAMPION FLATTENER ==================
def flatten_champion(champ, patch):
    row = {
        "patch": patch,
        "id": champ.get("id"),
        "key": champ.get("key"),
        "name": champ.get("name"),
        "title": champ.get("title"),
        "partype": champ.get("partype"),
        "tags": dump(champ.get("tags")),
        "lore": champ.get("lore"),
        "blurb": champ.get("blurb"),
        "allytips": dump(champ.get("allytips")),
        "enemytips": dump(champ.get("enemytips")),
        "skins": dump(champ.get("skins")),
        "skins_count": len(champ.get("skins", [])),
    }

    # info
    for k, v in champ.get("info", {}).items():
        row[f"info_{k}"] = v

    # stats
    for k, v in champ.get("stats", {}).items():
        row[f"stats_{k}"] = v

    # passive
    passive = champ.get("passive", {})
    row["passive_name"] = passive.get("name")
    row["passive_description"] = passive.get("description")
    row["passive_image"] = dump(passive.get("image"))

    # spells
    spells = champ.get("spells", [])
    labels = ["q", "w", "e", "r"]

    for i, label in enumerate(labels):
        if i < len(spells):
            row.update(flatten_spell(spells[i], f"spell_{label}"))
        else:
            row[f"spell_{label}_missing"] = True

    return row

# ================== PATCH PROCESSOR ==================
def process_patch(patch):
    print(f"\n=== PATCH {patch} ===")

    tgz_path = os.path.join(TGZ_DIR, f"dragontail-{patch}.tgz")
    csv_path = os.path.join(CSV_DIR, f"champions_{patch}.csv")
    patch_json_dir = os.path.join(JSON_DIR, patch)
    os.makedirs(patch_json_dir, exist_ok=True)

    # ---- DOWNLOAD IF NEEDED ----
    if not os.path.exists(tgz_path):
        print("⬇ Downloading tgz...")
        url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{patch}.tgz"
        r = requests.get(url)
        r.raise_for_status()
        with open(tgz_path, "wb") as f:
            f.write(r.content)
    else:
        print("✔ Using cached tgz")

    # ---- OPEN ARCHIVE ----
    with tarfile.open(tgz_path, "r:gz") as tar:
        prefix = f"{patch}/data/{LANG}/champion/"
        members = [
            m for m in tar.getmembers()
            if m.name.startswith(prefix) and m.name.endswith(".json")
        ]

        print(f"✔ {len(members)} champions")

        rows = []

        for m in members:
            try:
                raw = json.load(tar.extractfile(m))
                champ = next(iter(raw["data"].values()))
            except Exception as e:
                print("❌ JSON error:", m.name, e)
                continue

            # save raw json
            with open(
                os.path.join(patch_json_dir, os.path.basename(m.name)),
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)

            rows.append(flatten_champion(champ, patch))

    # ---- WRITE CSV ----
    if rows:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

    print(f"✅ CSV written: {csv_path}")

# ================== RUN ==================
if __name__ == "__main__":
    PATCHES = [
        "15.24.1",
        "15.23.1",
        "15.22.1",
        "15.21.1",
        "15.20.1",
        "15.19.1",
        "15.18.1",
        "15.17.1",
        "15.16.1",
        "15.15.1",
        "15.14.1",
        "15.13.1",
        "15.12.1",
        "15.11.1",
        "15.10.1",
        "15.9.1",
        "15.8.1",
        "15.7.1",
        "15.6.1",
        "15.5.1",
        "15.4.1",
        "15.3.1",
        "15.2.1",
        "15.1.1",
        "14.24.1",
    ]

    for patch in PATCHES:
        process_patch(patch)

    print("\n🎉 ALL PATCHES DONE")



=== PATCH 15.24.1 ===
✔ Using cached tgz
✔ 172 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.24.1.csv

=== PATCH 15.23.1 ===
✔ Using cached tgz
✔ 172 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.23.1.csv

=== PATCH 15.22.1 ===
✔ Using cached tgz
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.22.1.csv

=== PATCH 15.21.1 ===
✔ Using cached tgz
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.21.1.csv

=== PATCH 15.20.1 ===
✔ Using cached tgz
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.20.1.csv

=== PATCH 15.19.1 ===
✔ Using cached tgz
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.19.1.csv

=== PATCH 15.18.1 ===
✔ Using cached tgz
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.18.1.csv

ChunkedEncodingError: ("Connection broken: ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None)", ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))

In [3]:
import os
import pandas as pd
import re

# ================== CONFIG ==================
CSV_DIR = r"D:\lol draft analyzer\part 2\data champions\csv"
OUTPUT_DIR = CSV_DIR
# ============================================

def extract_patch(filename):
    """
    Extrait le patch depuis champions_15.24.1.csv
    """
    match = re.search(r"champions_(\d+\.\d+\.\d+)\.csv", filename)
    return match.group(1) if match else None

def aggregate_csvs():
    csv_files = [
        f for f in os.listdir(CSV_DIR)
        if f.startswith("champions_") and f.endswith(".csv")
    ]

    if not csv_files:
        raise RuntimeError("❌ Aucun CSV trouvé")

    dfs = []
    patches = []

    for file in csv_files:
        patch = extract_patch(file)
        if not patch:
            print("⚠ Fichier ignoré:", file)
            continue

        path = os.path.join(CSV_DIR, file)
        print("✔ Loading:", file)

        df = pd.read_csv(path)
        dfs.append(df)
        patches.append(patch)

    # concat
    final_df = pd.concat(dfs, ignore_index=True)

    # tri par patch + champion
    final_df.sort_values(by=["patch", "id"], inplace=True)

    # nom du fichier final
    patches_sorted = sorted(
        patches,
        key=lambda p: tuple(map(int, p.split(".")))
    )

    first_patch = patches_sorted[0]
    last_patch = patches_sorted[-1]

    output_name = f"champions_{first_patch}_{last_patch}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    final_df.to_csv(output_path, index=False, encoding="utf-8")

    print("\n🎉 AGGREGATION TERMINÉE")
    print("➡ Fichier généré :", output_path)
    print("➡ Lignes :", len(final_df))
    print("➡ Champions uniques :", final_df['id'].nunique())
    print("➡ Patchs :", final_df['patch'].nunique())

if __name__ == "__main__":
    aggregate_csvs()


✔ Loading: champions_15.1.1.csv
✔ Loading: champions_15.10.1.csv
✔ Loading: champions_15.11.1.csv
✔ Loading: champions_15.12.1.csv
✔ Loading: champions_15.13.1.csv
✔ Loading: champions_15.14.1.csv
✔ Loading: champions_15.15.1.csv
✔ Loading: champions_15.16.1.csv
✔ Loading: champions_15.17.1.csv
✔ Loading: champions_15.18.1.csv
⚠ Fichier ignoré: champions_15.18.1_15.24.1.csv
✔ Loading: champions_15.19.1.csv
✔ Loading: champions_15.2.1.csv
✔ Loading: champions_15.20.1.csv
✔ Loading: champions_15.21.1.csv
✔ Loading: champions_15.22.1.csv
✔ Loading: champions_15.23.1.csv
✔ Loading: champions_15.24.1.csv
✔ Loading: champions_15.3.1.csv
✔ Loading: champions_15.4.1.csv
✔ Loading: champions_15.5.1.csv
✔ Loading: champions_15.6.1.csv
✔ Loading: champions_15.7.1.csv
✔ Loading: champions_15.8.1.csv
✔ Loading: champions_15.9.1.csv
⚠ Fichier ignoré: champions_cleaned_15.18.1_15.24.1.csv

🎉 AGGREGATION TERMINÉE
➡ Fichier généré : D:\lol draft analyzer\part 2\data champions\csv\champions_15.1.1_15.24

In [ ]:
import pandas as pd

jhkhkhkhkCSV_PATH = r"D:\lol draft analyzer\part 2\data champions\csv\champions_15.1.1_15.24.1.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)


Shape: (4092, 120)


In [6]:
print(df.head())
print(df.info())
print(df.describe())
print(df.describe(include="object"))
print(df.columns.tolist())


    patch       id  key     name                 title        partype  \
0  15.1.1   Aatrox  266   Aatrox       Épée des Darkin  Puits de sang   
1  15.1.1     Ahri  103     Ahri  Renard à neuf queues           Mana   
2  15.1.1    Akali   84    Akali      Assassin rebelle        Énergie   
3  15.1.1   Akshan  166   Akshan    Sentinelle rebelle           Mana   
4  15.1.1  Alistar   12  Alistar             Minotaure           Mana   

                       tags  \
0               ["Fighter"]   
1      ["Mage", "Assassin"]   
2              ["Assassin"]   
3  ["Marksman", "Assassin"]   
4       ["Tank", "Support"]   

                                                lore  \
0  Autrefois, Aatrox et ses frères étaient honoré...   
1  Connectée à la magie du royaume spirituel, Ahr...   
2  Ayant abandonné l'Ordre Kinkou et le titre de ...   
3  Se jouant du danger, Akshan combat le mal sans...   
4  Alistar est un guerrier redoutable cherchant à...   

                                     

In [7]:
COLUMNS_TO_DROP = [
    "skins_count",
    "skins",
    "lore",
    "id",
    "title",
    "blurb",
    "allytips",
    "enemytips",
    "spell_q_description",
    "spell_w_description",
    "spell_e_description",
    "spell_r_description",
    "passive_description",
    "passive_name",
    "spell_q_name",
    "spell_w_name",
    "spell_e_name",
    "spell_r_name",
    "spell_r_tooltip",
    "spell_r_leveltip",
    "spell_e_tooltip",
    "spell_e_leveltip",
    "spell_w_tooltip",
    "spell_w_leveltip",
    "spell_q_tooltip",
    "spell_q_leveltip",
    "spell_q_image",
    "spell_w_image",
    "spell_e_image",
    "spell_r_image",
    "passive_image",

    "spell_q_cooldownBurn",
    "spell_q_costBurn",
    "spell_q_effectBurn",
    "spell_q_rangeBurn",
    "spell_w_cooldownBurn",
    "spell_w_costBurn",
    "spell_w_effectBurn",
    "spell_w_rangeBurn",
    "spell_e_cooldownBurn",
    "spell_e_costBurn",
    "spell_e_effectBurn",
    "spell_e_rangeBurn",
    "spell_r_cooldownBurn",
    "spell_r_costBurn",
    "spell_r_effectBurn",
    "spell_r_rangeBurn",
]

df_clean = df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [8]:
OUTPUT_PATH = r"D:\lol draft analyzer\part 2\data champions\csv\champions_cleaned_15.1.1_15.24.1.csv"

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("✅ CSV nettoyé sauvegardé :", OUTPUT_PATH)

✅ CSV nettoyé sauvegardé : D:\lol draft analyzer\part 2\data champions\csv\champions_cleaned_15.1.1_15.24.1.csv


In [13]:
counts = df_clean["name"].value_counts()

champions_less_than_3 = counts[counts < 24]

print(champions_less_than_3)

name
Mel       23
Yunara    11
Zaahen     2
Name: count, dtype: int64


In [16]:
df_clean.tail(10)

,patch,key,name,partype,tags,info_attack,info_defense,info_magic,info_difficulty,stats_hp,...,spell_r_maxrank,spell_r_cooldown,spell_r_cost,spell_r_datavalues,spell_r_effect,spell_r_vars,spell_r_costType,spell_r_maxammo,spell_r_range,spell_r_resource
4082,15.9.1,777,Yone,Impulsion,"[""Fighter"", ""Assassin""]",8,4,4,8,620,...,3,"[120, 100, 80]","[0, 0, 0]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],Pas de coût,-1,"[1000, 1000, 1000]",Pas de coût
4083,15.9.1,83,Yorick,Mana,"[""Fighter"", ""Tank""]",6,6,4,6,650,...,3,"[160, 130, 100]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[600, 600, 600]",{{ cost }} {{ abilityresourcename }}
4084,15.9.1,350,Yuumi,Mana,"[""Support"", ""Mage""]",5,1,8,2,500,...,3,"[120, 110, 100]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[1100, 1100, 1100]",{{ cost }} {{ abilityresourcename }}
4085,15.9.1,154,Zac,Aucune,"[""Tank"", ""Fighter""]",3,7,7,8,685,...,3,"[120, 105, 90]","[0, 0, 0]",{},"[null, [150, 250, 350], [1.1, 1.1, 1.1], [700,...",[],Pas de coût,-1,"[300, 300, 300]",Pas de coût
4086,15.9.1,238,Zed,Énergie,"[""Assassin""]",9,2,1,7,654,...,3,"[120, 110, 100]","[0, 0, 0]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],Pas de coût,-1,"[625, 625, 625]",Pas de coût
4087,15.9.1,221,Zeri,Mana,"[""Marksman""]",8,5,3,6,600,...,3,"[80, 75, 70]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[800, 800, 800]",{{ cost }} {{ abilityresourcename }}
4088,15.9.1,115,Ziggs,Mana,"[""Mage""]",2,4,9,4,606,...,3,"[120, 95, 70]","[100, 100, 100]",{},"[null, [300, 450, 600], [66.6667, 66.6667, 66....",[],{{ abilityresourcename }},-1,"[5000, 5000, 5000]",{{ cost }} {{ abilityresourcename }}
4089,15.9.1,26,Zilean,Mana,"[""Support"", ""Mage""]",2,5,8,6,574,...,3,"[120, 90, 60]","[125, 150, 175]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[900, 900, 900]",{{ cost }} {{ abilityresourcename }}
4090,15.9.1,142,Zoé,Mana,"[""Mage""]",1,7,8,5,630,...,3,"[11, 8, 5]","[40, 40, 40]",{},"[null, [-0.3, -0.4, -0.5], [1.5, 2, 2.5], [4, ...",[],{{ abilityresourcename }},-1,"[575, 575, 575]",{{ cost }} {{ abilityresourcename }}
4091,15.9.1,143,Zyra,Mana,"[""Mage"", ""Support""]",4,3,8,7,574,...,3,"[110, 100, 90]","[100, 100, 100]",{},"[null, [1, 1, 1], [50, 50, 50], [180, 265, 350...",[],{{ abilityresourcename }},-1,"[700, 700, 700]",{{ cost }} {{ abilityresourcename }}


In [1]:
df_clean.loc[df_clean["name"] == "Teemo", ["spell_q_effect"]].iloc[0]
# .to_string(index=False)



NameError: name 'df_clean' is not defined

In [28]:
pd.set_option("display.max_colwidth", None)

print(
    df_clean.loc[df_clean["name"] == "Teemo", ["spell_q_effect"]]
    .iloc[0]
    .to_string()
)

spell_q_effect    [null, [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [29]:
pd.set_option("display.max_colwidth", None)

print(
    df_clean.loc[df_clean["name"] == "Quinn", ["spell_q_effect"]]
    .iloc[0]
    .to_string()
)

spell_q_effect    [null, [20, 40, 60, 80, 100], [-1000, -1000, -1000, -1000, -1000], [1.75, 1.75, 1.75, 1.75, 1.75], [0.8, 0.9, 1, 1.1, 1.2], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [30]:
pd.set_option("display.max_colwidth", None)

print(
    df_clean.loc[df_clean["name"] == "Malphite", ["spell_r_effect"]]
    .iloc[0]
    .to_string()
)

spell_r_effect    [null, [1.5, 1.75, 2], [200, 300, 400], [1.5, 1.5, 1.5], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]


In [34]:
pd.set_option("display.max_colwidth", None)

print(
    df_clean.loc[df_clean["name"] == "Fiddlesticks", ["spell_q_effect"]]
    .iloc[0]
    .to_string()
)

spell_q_effect    [null, [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]


In [5]:
import pandas as pd
INPUT_PATH = r"D:\lol draft analyzer\part 2\data champions\csv\champions_cleaned_15.1.1_15.24.1.csv"

df_champs= pd.read_csv(INPUT_PATH, encoding="utf-8")

In [6]:
df_champs.head()

,patch,key,name,partype,tags,info_attack,info_defense,info_magic,info_difficulty,stats_hp,...,spell_r_maxrank,spell_r_cooldown,spell_r_cost,spell_r_datavalues,spell_r_effect,spell_r_vars,spell_r_costType,spell_r_maxammo,spell_r_range,spell_r_resource
0,15.1.1,266,Aatrox,Puits de sang,"[""Fighter""]",8,4,3,4,650,...,3,"[120, 100, 80]","[0, 0, 0]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],Pas de coût,-1,"[25000, 25000, 25000]",Pas de coût
1,15.1.1,103,Ahri,Mana,"[""Mage"", ""Assassin""]",3,4,8,5,590,...,3,"[130, 115, 100]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[450, 450, 450]",{{ cost }} {{ abilityresourcename }}
2,15.1.1,84,Akali,Énergie,"[""Assassin""]",5,3,8,7,600,...,3,"[120, 90, 60]","[0, 0, 0]",{},"[null, [0, 0, 0], [0, 0, 0], [1, 1, 1], [0, 0,...",[],Pas de coût,-1,"[675, 675, 675]",Pas de coût
3,15.1.1,166,Akshan,Mana,"[""Marksman"", ""Assassin""]",0,0,0,0,630,...,3,"[100, 85, 70]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[2500, 2500, 2500]",{{ cost }} {{ abilityresourcename }}
4,15.1.1,12,Alistar,Mana,"[""Tank"", ""Support""]",6,9,5,7,685,...,3,"[120, 100, 80]","[100, 100, 100]",{},"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...",[],{{ abilityresourcename }},-1,"[1, 1, 1]",{{ cost }} {{ abilityresourcename }}


In [7]:
df_champs.columns.tolist()

['patch',
 'key',
 'name',
 'partype',
 'tags',
 'info_attack',
 'info_defense',
 'info_magic',
 'info_difficulty',
 'stats_hp',
 'stats_hpperlevel',
 'stats_mp',
 'stats_mpperlevel',
 'stats_movespeed',
 'stats_armor',
 'stats_armorperlevel',
 'stats_spellblock',
 'stats_spellblockperlevel',
 'stats_attackrange',
 'stats_hpregen',
 'stats_hpregenperlevel',
 'stats_mpregen',
 'stats_mpregenperlevel',
 'stats_crit',
 'stats_critperlevel',
 'stats_attackdamage',
 'stats_attackdamageperlevel',
 'stats_attackspeedperlevel',
 'stats_attackspeed',
 'spell_q_id',
 'spell_q_maxrank',
 'spell_q_cooldown',
 'spell_q_cost',
 'spell_q_datavalues',
 'spell_q_effect',
 'spell_q_vars',
 'spell_q_costType',
 'spell_q_maxammo',
 'spell_q_range',
 'spell_q_resource',
 'spell_w_id',
 'spell_w_maxrank',
 'spell_w_cooldown',
 'spell_w_cost',
 'spell_w_datavalues',
 'spell_w_effect',
 'spell_w_vars',
 'spell_w_costType',
 'spell_w_maxammo',
 'spell_w_range',
 'spell_w_resource',
 'spell_e_id',
 'spell_e_max

In [ ]:
for col in df_champs.columns:
    print("Column:", col)
    print(df_champs[col].unique())
    print(len(df_champs[col].unique()))

Column: patch
['15.1.1' '15.10.1' '15.11.1' '15.12.1' '15.13.1' '15.14.1' '15.15.1'
 '15.16.1' '15.17.1' '15.18.1' '15.19.1' '15.2.1' '15.20.1' '15.21.1'
 '15.22.1' '15.23.1' '15.24.1' '15.3.1' '15.4.1' '15.5.1' '15.6.1'
 '15.7.1' '15.8.1' '15.9.1']
24
Column: key
[266 103  84 166  12 799  32  34   1 523  22 136 893 268 432 200  53  63
 201 233  51 164  69  31  42 122 131  36 119 245  60  28  81   9 114 105
   3  41  86 150  79 104 887 120  74 910 420  39 427  40  59  24 126 202
 222 897 145 429  43  30  38  55  10 141  85 121 203 240  96   7  64  89
 876 127 236 117  99  54  90  57  11 902  21  62  82  25 950 267  75 111
 518  76 895  56  20   2  61 516  80  78 555 246 133 497  33 421 526 888
  58 107  92  68  13 360 113 235 147 875  35  98 102  27  14  15  72 901
  37  16  50 517 134 223 163  91  44  17 412  18  48  23   4  29  77   6
 110  67  45 161 711 254 234 112   8 106  19 498 101   5 157 777  83 350
 154 238 221 115  26 142 143 800 804 904]
172
Column: name
['Aatrox' 'Ahri' 'A

: 